In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key not set (and this is optional)
Grok API Key not set (and this is optional)


In [3]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)


In [4]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8250U CPU @ 1.60GHz',
  'cores_logical': 8,
  'cores_physical': 0,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Short answer: you don’t currently have a C++ compiler installed. You’ll need to install one.

Simplest setup on your Windows 11 machine (using winget)
1) Open Windows PowerShell as Administrator.
2) Install the Visual C++ Build Tools (MSVC) with the C++ workload:
   winget install --id Microsoft.VisualStudio.2022.BuildTools -e --override "--quiet --wait --norestart --add Microsoft.VisualStudio.Workload.VCTools --includeRecommended"
   - This installs cl (the MSVC compiler), the MSVC STL, and the Windows SDK.

Python commands to compile and run main.cpp (fastest runtime focus)
Assuming default install path for Build Tools and that main.cpp is in your current working directory:

compile_command = [
    "cmd.exe", "/d", "/c",
    "\"C:\\Program Files\\Microsoft Visual Studio\\2022\\BuildTools\\Common7\\Tools\\VsDevCmd.bat\" -arch=x64 && cl /nologo /std:c++20 /O2 /GL /arch:AVX2 /fp:fast /DNDEBUG main.cpp /Fe:main.exe /link /LTCG"
]

run_command = ["main.exe"]

Notes:
- The VsDevCmd.bat call sets up the MSVC environment for that compile process.
- Flags chosen for speed: /O2 (optimize), /GL + /LTCG (link-time optimization), /arch:AVX2 (uses your CPU’s AVX2), /fp:fast (faster floating point; remove if you need strict FP accuracy), /DNDEBUG (disables assertions).
- If you installed to a non-default path, adjust the VsDevCmd.bat path accordingly.

In [17]:
import subprocess
import os

work_dir = r"C:\Users\prate\Repositories\AI\AI_Study\AI_Code_Converter"
vsdevcmd = r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\Common7\Tools\VsDevCmd.bat"

# Build the inner command as a single string for shell=True
compile_command = (
    f'"{vsdevcmd}" -arch=x64 && '
    f'cd /d "{work_dir}" && '
    'cl /nologo /std:c++20 /O2 /GL /arch:AVX2 /fp:fast /EHsc /DNDEBUG main.cpp /Fe:main.exe /link /LTCG'
)

run_command = os.path.join(work_dir, "main.exe")

# --- compile ---
print("Compiling...")
result = subprocess.run(compile_command, capture_output=True, text=True, shell=True, cwd=work_dir)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Compilation failed")
print("Compilation successful!")

# --- run ---
print("Running:")
print(subprocess.run(run_command, capture_output=True, text=True).stdout)


Compiling...
**********************************************************************
** Visual Studio 2022 Developer Command Prompt v17.0
** Copyright (c) 2025 Microsoft Corporation
**********************************************************************
main.cpp
Generating code
Finished generating code

Compilation successful!
Running:
Hello from C++!

